# 🎤 Entrena tu voz para VibeMusic

Este cuaderno crea un **modelo de tu voz** (RVC, con [Applio](https://github.com/IAHispano/Applio) 3.6.6) usando la GPU gratuita de Google Colab o Kaggle. Lo haces **una sola vez**: después VibeMusic convierte las voces de tus canciones a tu timbre en tu propio equipo.

**Antes de empezar**
1. Graba de **10 a 30 minutos** de tu voz, mejor cantando (aunque sea mal): sin música de fondo, sin eco, en un sitio silencioso. Más minutos limpios = más parecido.
2. Mete todos los archivos (WAV, MP3, FLAC…) en **un .zip**.
3. Activa la GPU: **Colab** → *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*. **Kaggle** → *Settings → Accelerator → GPU T4* e *Internet: On*.

Luego ejecuta las celdas **en orden** (▶). Entrenar tarda entre 30 y 90 minutos.

> Usa solo tu propia voz o la de alguien que te haya dado permiso.

## 1. Ajustes

In [ ]:
# Nombre de tu voz (sin espacios raros). Así aparecerá en VibeMusic.
NOMBRE = "mi_voz"

# Épocas de entrenamiento: 200–300 suele bastar con 10–30 min de audio.
EPOCAS = 250

# Frecuencia de muestreo del modelo (40000 es lo habitual).
SAMPLE_RATE = "40000"

## 2. Instalar Applio (2–4 min)

In [ ]:
import os, sys, subprocess
EN_KAGGLE = os.path.exists("/kaggle")
BASE = "/kaggle/working" if EN_KAGGLE else "/content"
os.chdir(BASE)
if not os.path.exists("Applio"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "3.6.6",
                    "https://github.com/IAHispano/Applio.git"], check=True)
os.chdir(f"{BASE}/Applio")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-engine.txt"], check=True)
subprocess.run([sys.executable, "core.py", "prerequisites", "--pretraineds-hifigan", "--models", "--no-exe"], check=True)
print("✓ Applio listo")

## 3. Sube tus grabaciones

- **Colab**: al ejecutar la celda aparece un botón para elegir tu .zip.
- **Kaggle**: sube el .zip como *Dataset* (*Add Input*) y escribe su ruta en `ZIP_KAGGLE` (por ejemplo `/kaggle/input/mi-voz/grabaciones.zip`).

In [ ]:
import zipfile, glob, shutil
ZIP_KAGGLE = ""  # solo en Kaggle

DATASET = f"{BASE}/dataset/{NOMBRE}"
shutil.rmtree(DATASET, ignore_errors=True)
os.makedirs(DATASET, exist_ok=True)
if EN_KAGGLE:
    zip_path = ZIP_KAGGLE
else:
    from google.colab import files
    subido = files.upload()
    zip_path = f"{BASE}/{next(iter(subido))}"
    shutil.move(next(iter(subido)), zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(DATASET)
audios = [f for f in glob.glob(f"{DATASET}/**/*", recursive=True)
          if f.lower().endswith((".wav", ".mp3", ".flac", ".ogg", ".m4a"))]
print(f"✓ {len(audios)} archivos de audio")
assert audios, "El .zip no tiene audios"

## 4. Preparar y extraer rasgos (5–15 min)

In [ ]:
def applio(*args):
    print("$ core.py", *args)
    subprocess.run([sys.executable, "core.py", *args], check=True)

applio("preprocess", "--model-name", NOMBRE, "--dataset-path", DATASET,
       "--sample-rate", SAMPLE_RATE, "--cpu-cores", str(os.cpu_count() or 2),
       "--cut-preprocess", "Automatic")
applio("extract", "--model-name", NOMBRE, "--f0-method", "rmvpe", "--gpu", "0",
       "--sample-rate", SAMPLE_RATE, "--cpu-cores", str(os.cpu_count() or 2))
print("✓ Datos preparados")

## 5. Entrenar (30–90 min)

No cierres la pestaña. Si Colab se desconecta, vuelve a ejecutar desde la celda 2: retoma lo guardado.

In [ ]:
applio("train", "--model-name", NOMBRE, "--save-every-epoch", "25", "--save-only-latest",
       "--total-epoch", str(EPOCAS), "--sample-rate", SAMPLE_RATE, "--batch-size", "8",
       "--gpu", "0", "--index-algorithm", "Auto")
print("✓ Entrenamiento terminado")

## 6. Descargar tu voz

Genera `NOMBRE.zip` con el modelo (`.pth`) y su índice (`.index`). Impórtalo en VibeMusic: **Ajustes → Mi voz → Importar voz**.

In [ ]:
LOGS = f"{BASE}/Applio/logs/{NOMBRE}"
pesos = [p for p in glob.glob(f"{LOGS}/**/*.pth", recursive=True)
         if not os.path.basename(p).startswith(("G_", "D_"))]
indices = glob.glob(f"{LOGS}/**/*.index", recursive=True)
assert pesos, "No se encontró el modelo entrenado: revisa la celda 5"
modelo = max(pesos, key=os.path.getmtime)
indice = max(indices, key=lambda p: ("added" in p, os.path.getmtime(p))) if indices else None

salida = f"{BASE}/{NOMBRE}.zip"
with zipfile.ZipFile(salida, "w") as z:
    z.write(modelo, f"{NOMBRE}.pth")
    if indice:
        z.write(indice, f"{NOMBRE}.index")
print("✓ Listo:", salida, "· modelo:", os.path.basename(modelo), "· índice:", bool(indice))
if EN_KAGGLE:
    print("En Kaggle: descárgalo desde el panel Output (a la derecha).")
else:
    from google.colab import files
    files.download(salida)